Copyright **`(c)`** 2025 Giovanni Squillero `<giovanni.squillero@polito.it>`  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free under certain conditions — see the [`license`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE.md) for details.  

In [7]:
from itertools import product, combinations
import numpy as np
import networkx as nx
from icecream import ic

In [8]:
def create_problem(
    size: int,
    *,
    density: float = 1.0,
    negative_values: bool = False,
    noise_level: float = 0.0,
    seed: int = 42,
) -> np.ndarray:
    """Problem generator for Lab3"""
    rng = np.random.default_rng(seed)
    # Genera N (size) punti casuali nel piano
    map = rng.random(size=(size, 2))
    # Inizializza la matrice dei pesi casuali
    problem = rng.random((size, size))
    if negative_values:
        problem = problem * 2 - 1
    problem *= noise_level
    for a, b in product(range(size), repeat=2):
        if rng.random() < density:
            # se l'arco esiste distanza euclidea + rumore (dell'inizializzazione)
            problem[a, b] += np.sqrt(
                np.square(map[a, 0] - map[b, 0]) + np.square(map[a, 1] - map[b, 1])
            )
        else:
            # se l'arco non esiste peso infinito
            problem[a, b] = np.inf
    np.fill_diagonal(problem, 0)
    return (problem * 1_000).round()


# Crea N punti in un piano 2D.
#Calcola le loro distanze euclidee.
#Usa queste distanze come pesi degli archi del grafo.
#Con density < 1, alcuni archi vengono rimossi (posti a inf → nessun collegamento).

#Aggiunge eventualmente rumore e valori negativi.

In [26]:
problem = create_problem(50, density=0.15, noise_level=10, negative_values=True)
problem

array([[    0.,    inf,    inf, ...,    inf,    inf,    inf],
       [   inf,     0.,    inf, ...,  8052.,    inf,    inf],
       [   inf,    inf,     0., ...,    inf,    inf, -1455.],
       ...,
       [   inf,    inf,    inf, ...,     0.,    inf,    inf],
       [   inf,    inf,    inf, ...,    inf,     0.,    inf],
       [   inf,    inf,    inf, ...,    inf,    inf,     0.]],
      shape=(50, 50))

In [11]:
masked = np.ma.masked_array(problem, mask=np.isinf(problem))
G = nx.from_numpy_array(masked, create_using=nx.DiGraph)

In [ ]:
for s, d in combinations(range(problem.shape[0]), 2):
    try:
        # path = nx.shortest_path(G, s, d, weight='weight')
        path = nx.bellman_ford_path(G, s, d, weight='weight')
        cost = cost = nx.path_weight(G, path, weight='weight')
    except nx.NetworkXNoPath:
        # Nodes are not connected
        path = None
        cost = np.inf
    except nx.NetworkXUnbounded:
        # Negative cycle detected
        path = None
        cost = -np.inf
    ic(s, d, path, cost)
None

In [39]:
#Breadth-first search

# Like Breadth-first, but the node with the lowest path cost (lower cost) from the root is expanded, 
# The frontier is a real priority queue, first node expanded is the closest from the actual node, 
# Also called Dijkstra’s algorithm. 

def bfs(graph: np.ndarray, start: int, goal: int) -> tuple[list[int], float]:
    
    size = graph.shape[0]
    visited = [False] * size
    parent = [None] * size
    # ( nodo, costo accumulato )
    frontier = [(start, 0.0)]
    
    visited[start] = True

    while frontier:
        ##ic(frontier)
        current_node, current_cost = frontier.pop(0)

        if current_node == goal:
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = parent[current_node]
            return path[::-1], current_cost

        for neighbor in range(size):
            weight = graph[current_node, neighbor]
            if weight != np.inf and not visited[neighbor]:
                visited[neighbor] = True
                parent[neighbor] = current_node
                
                frontier.append((neighbor, current_cost + weight))
        # ordina la coda in base al costo accumulato (crescente)
    
        

    return None, np.inf


s, d = 0, 19
path, cost = bfs(problem, s, d)
ic(s, d, path, cost)

ic| s: 0, d: 19, path: [0, 5, 19], cost: np.float64(13233.0)


(0, 19, [0, 5, 19], np.float64(13233.0))

In [ ]:
import numpy as np

def best_fit(graph: np.ndarray, start: int, goal: int) -> tuple[list[int], float]:
    
    size = graph.shape[0]
    # ( nodo, costo accumulato )
    
    #esclude percorsi circolari
    # mantiene per ogni percorso i nodi visitati
    feasible_paths=[]
    frontier=[]
    path_id = 0
    for i in range(size):
        if i != start and graph[start, i] != np.inf:
            frontier.append((path_id, i, graph[start, i]))
            feasible_paths.append([start, i])
            path_id += 1

    
    #print("frontiera iniziale:", frontier)
    #print("path possibili iniziali: ", feasible_paths)
    goal_path=[]    

    while frontier:
        #print ("frontiera:", frontier)
        path_id, current_node, current_cost = frontier.pop(0)
        #print ("estraggo:", path_id, current_node, current_cost)
        if current_node == goal:
            
            path= feasible_paths[path_id]
            #print("goal reached", path)
            if path not in [p[0] for p in goal_path]:
             goal_path.append((path, current_cost))
             rate= len(goal_path)/len(feasible_paths)
             ic(path_id, path, current_cost,rate )
             if rate < 0.001:
                 break
            continue
            

        for neighbor in range(size):
            weight = graph[current_node, neighbor]
            if weight != np.inf and neighbor not in feasible_paths[path_id]:
                current_path= feasible_paths[path_id]
                new_path=current_path + [neighbor]
                ## evita percorsi uguali
                if new_path not in [p for p in feasible_paths]:
                    ##add new path
                    new_path_id= len(feasible_paths)
                    feasible_paths.append(new_path)

                    #print(new_path_id, feasible_paths[new_path_id])
                    frontier.append((new_path_id, neighbor, current_cost + weight))
                #else:
                    #print("ignoro nodo:", neighbor, " path:", current_path)
            #else:
                #print("ignoro nodo:", neighbor, "weight:", weight)
                
        # ordina la coda in base al costo accumulato (crescente)
        frontier.sort(key=lambda x: x[1])  # best-first
        
    if goal_path:
        # ordino in ordine crescente di lunghezza e a parità di lunghezza di costo
        return min(goal_path, key=lambda x: (len(x[0]), x[1]))
       
    return None, np.inf


s, d = 0, 19
path, cost = best_fit(problem, s, d)
print( "start:", s, " end:", d, " path:", path, " cost:", cost)

In [33]:
d=0
for s, d in combinations(range(problem.shape[0]), 2):
    p1, c1 =best_fit(problem, s, d)
    p2, c2 =bfs(problem, s, d)
    if p1 != p2 or c1 != c2:
        d+=1
        ic(f"{d} Discrepancy found:")
        ic(s, d)
        ic("Best-fit:", p1, c1)
        ic("Breadth first:", p2, c2)
    # else:
    #     ic(s, d, p1, c1)
print("Number of discrepancies:", d)
None

ic| f"{d} Discrepancy found:": '2 Discrepancy found:'
ic| s: 0, d: 2
ic| "Best-fit:": 'Best-fit:'
    p1: [0, 4, 14, 5, 13, 12, 9, 1]
    c1: np.float64(-26030.0)
ic| "Breadth first:": 'Breadth first:'
    p2: [0, 30, 46, 41, 28, 34, 36, 11, 12, 33, 1]
    c2: np.float64(-57262.0)
ic| f"{d} Discrepancy found:": '3 Discrepancy found:'
ic| s: 0, d: 3
ic| "Best-fit:": 'Best-fit:'
    p1: [0, 5, 13, 12, 9, 1, 8, 4, 2]
    c1: np.float64(-10570.0)
ic| 'Breadth first:', p2: [0, 30, 46, 41, 2], c2: np.float64(-10331.0)
ic| f"{d} Discrepancy found:": '4 Discrepancy found:'
ic| s: 0, d: 4
ic| 'Best-fit:', p1: [0, 4, 13, 12, 17, 3], c1: np.float64(-18370.0)
ic| 'Breadth first:', p2: [0, 30, 46, 41, 3], c2: np.float64(-10211.0)
ic| f"{d} Discrepancy found:": '5 Discrepancy found:'
ic| s: 0, d: 5
ic| "Best-fit:": 'Best-fit:'
    p1: [0, 5, 13, 12, 17, 8, 4]
    c1: np.float64(-17801.0)
ic| 'Breadth first:', p2: [0, 4], c2: np.float64(6221.0)
ic| f"{d} Discrepancy found:": '6 Discrepancy found:'
ic

KeyboardInterrupt: 